# Systematic Trading Strategies with Machine Learning  
## Coursework Project – Meta-Labeling for Trading Signal Filtering

---

# 1. Project Overview

The objective of this project is to build a **metamodel** on top of a provided primary trading signal across futures contracts from multiple asset classes.

The primary model generates daily trading signals in \(\{-1,0,+1\}\):
- \(+1\): long signal
- \(-1\): short signal
- \(0\): no position

Our goal is **not** to predict returns directly.  
Instead, we aim to estimate the probability that a given primary signal is worth taking under a triple-barrier labeling framework.

More formally, we seek to model:

\[
P(\text{Trade is profitable} \mid \text{Features})
\]

using machine learning techniques.

---

# 2. Project Pipeline

The project is structured into the following stages:

## Phase 1 — Data Preparation and Exploration
- Load and clean OHLCV data
- Load and align primary signals
- Verify data integrity and temporal consistency
- Explore the characteristics of each asset class

## Phase 2 — Feature Engineering
Construction of predictive features from market data, including:
- Technical indicators
- Volatility and momentum features
- Cross-sectional features
- Latent regime features (HMM/GMM)
- Additional engineered features

## Phase 3 — Triple-Barrier Labeling
Implementation of the triple-barrier method to define supervised learning targets:
- Profit-taking barrier
- Stop-loss barrier
- Maximum holding period

We will justify all parameter choices economically and statistically.

## Phase 4 — Model Development
We will compare several model families:
- Linear models
- Tree-based ensemble methods
- Neural networks

Hyperparameter tuning and robust validation procedures will be applied.

## Phase 5 — Feature Importance Analysis
We will study:
- Individual feature importance
- Cluster-level feature importance
- SHAP/permutation-based interpretability

## Phase 6 — Out-of-Sample Evaluation
Evaluation on a clean out-of-sample period using:
- Precision
- Recall
- F1-score
- ROC-AUC
- Confusion matrices
- Comparison against the raw primary signal

## Optional Phase 7 — Strategy Construction
(Optional competition track)

Construction of a position-sizing strategy using metamodel probabilities.

---

# 3. Methodological Principles

Throughout the project, particular attention will be paid to:
- Avoiding look-ahead bias
- Preventing data leakage
- Using time-aware validation procedures
- Ensuring reproducibility
- Maintaining economic interpretability of results

The emphasis of this coursework is placed on:
- methodological rigor,
- feature engineering quality,
- validation protocol,
- and critical analysis of results.

---

# 4. Asset Universe

The project covers futures contracts from three asset classes:

## Equity Index Futures
- ES1S — S&P 500
- NQ1S — Nasdaq 100
- FESX1S — Euro Stoxx 50

## Energy
- CL1S — WTI Crude Oil
- HO1S — Heating Oil
- RB1S — RBOB Gasoline
- NG1S — Natural Gas

## Metals
- GC1S — Gold
- SI1S — Silver
- HG1S — Copper
- PL1S — Platinum

---

# 5. Initial Setup

We begin by importing the required libraries and loading the datasets.

# Phase 1 — Data Preparation and Initial Exploration

In this first phase, we prepare the raw datasets for the meta-labeling pipeline.

We focus initially on the **Energy** asset class:

| Ticker | Commodity |
|---|---|
| CL1S | WTI Crude Oil |
| HO1S | Heating Oil |
| RB1S | RBOB Gasoline |
| NG1S | Natural Gas |

The objectives of this phase are:

1. Load the OHLCV dataset and the primary signals dataset.
2. Standardize dates, instrument names, and column formats.
3. Filter the universe to Energy instruments only.
4. Align market data and primary signals on a common trading calendar.
5. Perform basic data integrity checks:
   - missing values,
   - duplicated rows,
   - date coverage,
   - signal distribution,
   - price and volume sanity checks.
6. Prepare a clean base dataframe that will later be used for feature engineering and triple-barrier labeling.

At this stage, we do **not** create labels yet.  
The purpose is to build a clean, reliable data foundation before moving to feature engineering.

In [ ]:
# ============================================================
# Phase 1 — Setup and Data Loading
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Display options
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

# ------------------------------------------------------------
# Project configuration
# ------------------------------------------------------------

DATA_DIR = "."  # Change this path if your CSV files are in another folder

OHLCV_FILE = os.path.join(DATA_DIR, "ohlcv_data.csv")
SIGNALS_FILE = os.path.join(DATA_DIR, "primary_signals.csv")

ENERGY_INSTRUMENTS = ["cl1s", "ho1s", "rb1s", "ng1s"]

INSTRUMENT_NAMES = {
    "cl1s": "WTI Crude Oil",
    "ho1s": "Heating Oil",
    "rb1s": "RBOB Gasoline",
    "ng1s": "Natural Gas",
}

print("Energy universe:")
for ticker, name in INSTRUMENT_NAMES.items():
    print(f"- {ticker.upper()}: {name}")

In [ ]:
# ============================================================
# Load Raw Data
# ============================================================

ohlcv_raw = pd.read_csv(OHLCV_FILE)
signals_raw = pd.read_csv(SIGNALS_FILE)

print("OHLCV shape:", ohlcv_raw.shape)
print("Signals shape:", signals_raw.shape)

display(ohlcv_raw.head())
display(signals_raw.head())

## 1.1 Initial Data Standardization

We now standardize the two datasets:

- convert dates to `datetime`,
- convert instrument names to lowercase,
- sort observations chronologically,
- filter the OHLCV data and primary signals to the Energy instruments only.

This ensures that both datasets use the same naming convention and can later be merged safely.

In [ ]:
# ============================================================
# Standardize Dates and Instrument Names
# ============================================================

ohlcv = ohlcv_raw.copy()
signals = signals_raw.copy()

# Convert date columns
ohlcv["date"] = pd.to_datetime(ohlcv["date"])
signals["date"] = pd.to_datetime(signals["date"])

# Standardize instrument names
ohlcv["instrument"] = ohlcv["instrument"].str.lower()

# Keep only Energy instruments
ohlcv_energy = ohlcv[ohlcv["instrument"].isin(ENERGY_INSTRUMENTS)].copy()

# Keep date + Energy signal columns
signals_energy = signals[["date"] + ENERGY_INSTRUMENTS].copy()

# Sort
ohlcv_energy = ohlcv_energy.sort_values(["instrument", "date"]).reset_index(drop=True)
signals_energy = signals_energy.sort_values("date").reset_index(drop=True)

print("Energy OHLCV shape:", ohlcv_energy.shape)
print("Energy signals shape:", signals_energy.shape)

display(ohlcv_energy.head())
display(signals_energy.head())

## 1.2 Data Integrity Checks

Before creating features or labels, we first verify the quality of the raw Energy datasets.

We check:
- date coverage by instrument,
- duplicated observations,
- missing values,
- OHLCV consistency,
- and primary signal distributions.

These checks are important because any data issue at this stage could later create misleading labels, feature leakage, or incorrect model evaluation.

In [ ]:
# ============================================================
# Basic Data Integrity Checks
# ============================================================

print("OHLCV date range by instrument:")
display(
    ohlcv_energy
    .groupby("instrument")["date"]
    .agg(["min", "max", "count"])
)

print("\nSignals date range:")
display(
    signals_energy["date"].agg(["min", "max", "count"])
)

print("\nDuplicate OHLCV rows by (date, instrument):")
n_dup_ohlcv = ohlcv_energy.duplicated(subset=["date", "instrument"]).sum()
print(n_dup_ohlcv)

print("\nDuplicate signal dates:")
n_dup_signals = signals_energy.duplicated(subset=["date"]).sum()
print(n_dup_signals)

print("\nMissing values in OHLCV:")
display(ohlcv_energy.isna().sum())

print("\nMissing values in signals:")
display(signals_energy.isna().sum())

## 1.3 Signal Distribution Analysis

Before building labels or training models, we inspect the distribution of the primary signals.

This is important because:
- the dataset may be imbalanced,
- some instruments may trade much more frequently than others,
- and the proportion of long/short/flat signals may affect both labeling and model performance.

We therefore compute the distribution of:
- long signals (+1),
- short signals (-1),
- and inactive periods (0),
for each Energy instrument.

In [ ]:
# ============================================================
# Signal Distribution
# ============================================================

signal_distribution = {}

for inst in ENERGY_INSTRUMENTS:
    
    counts = (
        signals_energy[inst]
        .value_counts()
        .sort_index()
    )
    
    signal_distribution[inst] = counts

    print(f"\n{inst.upper()} signal distribution:")
    print(counts)

    print("\nPercentages:")
    print((counts / counts.sum() * 100).round(2))

## 1.4 OHLC Consistency Checks

We now perform several sanity checks on the OHLC data.

For each observation, financial market conventions imply:


$\text{low} \leq \text{open, close} \leq \text{high}$


Violations of these inequalities may indicate:
- corrupted observations,
- bad data adjustments,
- or preprocessing issues.

We also verify that prices and trading activity remain strictly positive.

In [ ]:
# ============================================================
# OHLC Consistency Checks
# ============================================================

# High should be >= low
invalid_high_low = (ohlcv_energy["high"] < ohlcv_energy["low"]).sum()

# Open should lie inside [low, high]
invalid_open = (
    (ohlcv_energy["open"] < ohlcv_energy["low"]) |
    (ohlcv_energy["open"] > ohlcv_energy["high"])
).sum()

# Close should lie inside [low, high]
invalid_close = (
    (ohlcv_energy["close"] < ohlcv_energy["low"]) |
    (ohlcv_energy["close"] > ohlcv_energy["high"])
).sum()

# Negative or zero prices
non_positive_prices = (
    (ohlcv_energy[["open", "high", "low", "close"]] <= 0)
    .sum()
    .sum()
)

print("Invalid high/low rows:", invalid_high_low)
print("Invalid open rows:", invalid_open)
print("Invalid close rows:", invalid_close)
print("Non-positive prices:", non_positive_prices)

## 1.5 Price Series Visualization

We now visualize the historical closing prices of the Energy futures contracts.

The objective is not yet predictive modeling, but rather:
- understanding the long-term behavior of each market,
- identifying volatility regimes,
- detecting structural breaks and crises,
- and building intuition about the data before feature engineering.

Since commodity futures can exhibit very different scales and volatility levels, each instrument is plotted separately.

In [ ]:
# ============================================================
# Historical Price Visualization
# ============================================================

fig, axes = plt.subplots(
    nrows=4,
    ncols=1,
    figsize=(10, 8),
    sharex=True
)

for ax, inst in zip(axes, ENERGY_INSTRUMENTS):

    df_inst = ohlcv_energy[
        ohlcv_energy["instrument"] == inst
    ]

    ax.plot(
        df_inst["date"],
        df_inst["close"],
        linewidth=1.2
    )

    ax.set_title(
        f"{inst.upper()} — {INSTRUMENT_NAMES[inst]}",
        fontsize=12
    )

    ax.set_ylabel("Close Price")

    ax.grid(True, alpha=0.3)

plt.xlabel("Date")
plt.tight_layout()
plt.show()

## 1.6 Return Series Construction

Financial machine learning models are generally built on returns rather than raw price levels.

We therefore compute daily log returns for each instrument:

$$
r_t = \log\left(\frac{P_t}{P_{t-1}}\right)
$$

Log returns are preferred because:
- they are additive through time,
- more statistically stable than prices,
- and largely invariant to the absolute price scale.

This is particularly important for futures contracts, where continuous-contract adjustments may distort long-term price levels.

In [ ]:
# ============================================================
# Daily Log Returns
# ============================================================

ohlcv_energy["log_return"] = (
    ohlcv_energy
    .groupby("instrument")["close"]
    .transform(lambda x: np.log(x / x.shift(1)))
)

display(
    ohlcv_energy[
        ["date", "instrument", "close", "log_return"]
    ].head(10)
)

In [ ]:
# ============================================================
# Log Return Visualization
# ============================================================

fig, axes = plt.subplots(
    nrows=4,
    ncols=1,
    figsize=(10, 8),
    sharex=True
)

for ax, inst in zip(axes, ENERGY_INSTRUMENTS):

    df_inst = ohlcv_energy[
        ohlcv_energy["instrument"] == inst
    ]

    ax.plot(
        df_inst["date"],
        df_inst["log_return"],
        linewidth=0.7
    )

    ax.set_title(
        f"{inst.upper()} Log Returns",
        fontsize=11
    )

    ax.axhline(0, color="black", linewidth=0.8)

    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 1.7 Reshaping Primary Signals

The primary signal dataset is currently stored in a wide format:

| date | cl1s | ho1s | rb1s | ng1s |
|---|---|---|---|---|

However, the OHLCV dataset is stored in long format:

| date | instrument | open | high | low | close | ... |

To simplify merging and downstream processing, we reshape the signal dataset into long format using `pandas.melt()`.

The resulting structure becomes:

| date | instrument | signal |
|---|---|---|

This representation is much more convenient for:
- feature engineering,
- labeling,
- merging datasets,
- and machine learning pipelines.

In [ ]:
# ============================================================
# Reshape Signals to Long Format
# ============================================================

signals_long = signals_energy.melt(
    id_vars="date",
    value_vars=ENERGY_INSTRUMENTS,
    var_name="instrument",
    value_name="primary_signal"
)

signals_long = (
    signals_long
    .sort_values(["instrument", "date"])
    .reset_index(drop=True)
)

print("Signals long shape:", signals_long.shape)

display(signals_long.head(10))

# Phase 2 — Feature Engineering

In this phase, we construct predictive features from the Energy OHLCV dataset.

The objective is to describe the market environment in which each primary signal occurs.  
These features will later be used by the metamodel to estimate whether a given primary trading signal is worth taking.

We construct features separately for each instrument, using only past and current information available at each date.  
This is essential to avoid look-ahead bias.

The feature engineering process is organized into several blocks:

1. **Core return and momentum features**
   - daily returns,
   - rolling cumulative returns,
   - short-term and medium-term momentum.

2. **Volatility features**
   - rolling realized volatility,
   - volatility ratios,
   - volatility regime indicators.

3. **Volume and open-interest features**
   - volume changes,
   - volume z-scores,
   - open-interest changes.

4. **Technical indicators**
   - RSI,
   - MACD,
   - Bollinger-style z-scores.

5. **Time-series dependence features**
   - rolling autocorrelation,
   - volatility clustering proxies,
   - trend persistence measures.

6. **Cross-sectional features**
   - relative momentum,
   - relative volatility,
   - ranks within the Energy asset class.

7. **Latent regime features**
   - GMM/HMM-based regime probabilities,
   - market turbulence indicators.

At this stage, we start with a robust set of core statistical features before adding more advanced indicators.

## 2.1 Core Return and Momentum Features

We begin with a set of core return-based and momentum features.

Momentum features aim to capture whether an instrument has recently experienced persistent positive or negative price movements.  
Such features are widely used in systematic trading strategies, where trend-following and persistence effects often play an important role.

For each instrument, we compute rolling momentum over several horizons:

$$
\text{mom}_{k,t}
=
\log\left(\frac{P_t}{P_{t-k}}\right)
$$

where:
- \(P_t\) is the closing price at time \(t\),
- and \(k\) represents the lookback horizon.

We use multiple horizons in order to capture:
- short-term momentum,
- medium-term trends,
- and longer-term price persistence.

In addition, we compute rolling average returns:

$$
\bar r_t^{(k)}
=
\frac{1}{k}
\sum_{i=0}^{k-1} r_{t-i}
$$

which provide smoother estimates of recent market direction and local return persistence.

All features are computed independently for each instrument using:

```python
groupby("instrument")

In [ ]:
# ============================================================
# Phase 2 — Feature Engineering Setup
# ============================================================

features_df = ohlcv_energy.copy()

# Make sure data is sorted before computing rolling/grouped features
features_df = (
    features_df
    .sort_values(["instrument", "date"])
    .reset_index(drop=True)
)

print("Feature engineering base shape:", features_df.shape)
display(features_df.head())

In [ ]:
# ============================================================
# Core Return and Momentum Features
# ============================================================

MOMENTUM_WINDOWS = [5, 10, 20, 60]

for window in MOMENTUM_WINDOWS:
    
    features_df[f"momentum_{window}d"] = (
        features_df
        .groupby("instrument")["close"]
        .transform(lambda x: np.log(x / x.shift(window)))
    )

# Rolling mean return features
RETURN_MEAN_WINDOWS = [5, 20, 60]

for window in RETURN_MEAN_WINDOWS:
    
    features_df[f"mean_return_{window}d"] = (
        features_df
        .groupby("instrument")["log_return"]
        .transform(lambda x: x.rolling(window=window).mean())
    )

display(
    features_df[
        ["date", "instrument", "close", "log_return",
         "momentum_5d", "momentum_20d", "momentum_60d",
         "mean_return_5d", "mean_return_20d"]
    ].head(25)
)

## 2.2 Volatility Features

Volatility is one of the most important variables in financial markets, especially for commodities and futures contracts.

Periods of high volatility often correspond to:
- market stress,
- regime shifts,
- liquidity shocks,
- or macroeconomic uncertainty.

We therefore construct several volatility-related features.

### Rolling Realized Volatility

We first compute rolling realized volatility using the standard deviation of daily log returns:

$$
\sigma_t^{(k)}
=
\sqrt{
\frac{1}{k-1}
\sum_{i=0}^{k-1}
(r_{t-i} - \bar r_t)^2
}
$$

over multiple horizons.

### EWMA Volatility

We also compute exponentially weighted volatility estimates, which react faster to recent market shocks.

### Volatility Ratios

Finally, we compute volatility ratios such as:

$$
\frac{\sigma_{20}}{\sigma_{60}}
$$

to detect volatility regime changes and volatility expansions.

In [ ]:
# ============================================================
# Rolling Realized Volatility
# ============================================================

VOL_WINDOWS = [5, 20, 60]

for window in VOL_WINDOWS:

    features_df[f"realized_vol_{window}d"] = (
        features_df
        .groupby("instrument")["log_return"]
        .transform(lambda x: x.rolling(window=window).std())
    )

display(
    features_df[
        [
            "date",
            "instrument",
            "log_return",
            "realized_vol_5d",
            "realized_vol_20d",
            "realized_vol_60d"
        ]
    ].head() #change to head(30) if you want to see more rows
)

In [ ]:
# ============================================================
# EWMA Volatility
# ============================================================

EWMA_SPANS = [10, 20]

for span in EWMA_SPANS:

    features_df[f"ewma_vol_{span}d"] = (
        features_df
        .groupby("instrument")["log_return"]
        .transform(
            lambda x: x.ewm(span=span, adjust=False).std()
        )
    )

display(
    features_df[
        [
            "date",
            "instrument",
            "ewma_vol_10d",
            "ewma_vol_20d"
        ]
    ].head() #change to head(30) if you want to see more rows
)

In [ ]:
# ============================================================
# Volatility Regime Features
# ============================================================

features_df["vol_ratio_20_60"] = (
    features_df["realized_vol_20d"]
    /
    features_df["realized_vol_60d"]
)

display(
    features_df[
        [
            "date",
            "instrument",
            "realized_vol_20d",
            "realized_vol_60d",
            "vol_ratio_20_60"
        ]
    ].head() #change to head(30) if you want to see more rows
)

## 2.3 Volume and Open Interest Features

Futures markets contain additional information beyond prices alone.

In particular:
- trading volume,
- and open interest

can provide insight into:
- market participation,
- liquidity conditions,
- trend confirmation,
- and abnormal trading activity.

We therefore construct several features based on:
- volume dynamics,
- abnormal volume detection,
- and changes in open interest.

### Volume Z-Scores

We compute rolling z-scores of volume:

$$
z_t^{vol}
=
\frac{V_t - \mu_t^{(k)}}{\sigma_t^{(k)}}
$$

to detect unusually high or low trading activity.

### Open Interest Changes

We also compute changes in open interest, which may help identify:
- trend strengthening,
- position buildup,
- or liquidation regimes.

All features are computed independently for each instrument.

In [ ]:
# ============================================================
# Volume Features
# ============================================================

# Log volume change
features_df["log_volume_change"] = (
    features_df
    .groupby("instrument")["volume"]
    .transform(lambda x: np.log(x / x.shift(1)))
)

# Rolling volume averages
VOLUME_WINDOWS = [5, 20]

for window in VOLUME_WINDOWS:

    features_df[f"volume_mean_{window}d"] = (
        features_df
        .groupby("instrument")["volume"]
        .transform(lambda x: x.rolling(window).mean())
    )

    features_df[f"volume_std_{window}d"] = (
        features_df
        .groupby("instrument")["volume"]
        .transform(lambda x: x.rolling(window).std())
    )

display(
    features_df[
        [
            "date",
            "instrument",
            "volume",
            "log_volume_change",
            "volume_mean_20d",
            "volume_std_20d"
        ]
    ].head() #change to head(30) if you want to see more rows
)

In [ ]:
# ============================================================
# Volume Z-Score Features
# ============================================================

features_df["volume_zscore_20d"] = (
    (
        features_df["volume"]
        - features_df["volume_mean_20d"]
    )
    /
    features_df["volume_std_20d"]
)

display(
    features_df[
        [
            "date",
            "instrument",
            "volume",
            "volume_mean_20d",
            "volume_std_20d",
            "volume_zscore_20d"
        ]
    ].head() #change to head(30) if you want to see more rows
)

In [ ]:
# ============================================================
# Open Interest Features
# ============================================================

# Log change in open interest
features_df["log_oi_change"] = (
    features_df
    .groupby("instrument")["open_interest"]
    .transform(lambda x: np.log(x / x.shift(1)))
)

# Open interest momentum
OI_WINDOWS = [5, 20]

for window in OI_WINDOWS:

    features_df[f"oi_momentum_{window}d"] = (
        features_df
        .groupby("instrument")["open_interest"]
        .transform(lambda x: np.log(x / x.shift(window)))
    )

display(
    features_df[
        [
            "date",
            "instrument",
            "open_interest",
            "log_oi_change",
            "oi_momentum_5d",
            "oi_momentum_20d"
        ]
    ].head() #change to head(30) if you want to see more rows
)

## 2.4 Technical Indicators

We now construct a set of classical technical indicators.

These indicators are widely used in systematic trading because they summarize different aspects of recent price dynamics:

- **RSI** captures overbought and oversold conditions.
- **Bollinger z-scores** measure how far the current price is from its recent moving average.
- **MACD** captures trend strength through the difference between short-term and long-term exponential moving averages.

Although these indicators are simple, they provide interpretable signals that can help the metamodel identify market environments where the primary signal is more or less reliable.

In [ ]:
# ============================================================
# RSI Indicator
# ============================================================

def compute_rsi(series, window=14):
    """
    Compute the Relative Strength Index (RSI).
    """
    delta = series.diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(window=window).mean()
    avg_loss = loss.rolling(window=window).mean()

    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))

    return rsi


features_df["rsi_14d"] = (
    features_df
    .groupby("instrument")["close"]
    .transform(lambda x: compute_rsi(x, window=14))
)

display(
    features_df[
        ["date", "instrument", "close", "rsi_14d"]
    ].head(70)
)

In [ ]:
# ============================================================
# Bollinger-Style Price Z-Score
# ============================================================

BOLLINGER_WINDOWS = [20, 60]

for window in BOLLINGER_WINDOWS:

    rolling_mean = (
        features_df
        .groupby("instrument")["close"]
        .transform(lambda x: x.rolling(window=window).mean())
    )

    rolling_std = (
        features_df
        .groupby("instrument")["close"]
        .transform(lambda x: x.rolling(window=window).std())
    )

    features_df[f"price_zscore_{window}d"] = (
        (features_df["close"] - rolling_mean) / rolling_std
    )

display(
    features_df[
        [
            "date",
            "instrument",
            "close",
            "price_zscore_20d",
            "price_zscore_60d"
        ]
    ].head(70)
)

In [ ]:
# ============================================================
# MACD Indicator
# ============================================================

def compute_macd(series, short_span=12, long_span=26, signal_span=9):
    """
    Compute MACD, MACD signal line, and MACD histogram.
    """
    ema_short = series.ewm(span=short_span, adjust=False).mean()
    ema_long = series.ewm(span=long_span, adjust=False).mean()

    macd = ema_short - ema_long
    macd_signal = macd.ewm(span=signal_span, adjust=False).mean()
    macd_hist = macd - macd_signal

    return macd, macd_signal, macd_hist


macd_results = (
    features_df
    .groupby("instrument")["close"]
    .apply(lambda x: pd.DataFrame({
        "macd": compute_macd(x)[0],
        "macd_signal": compute_macd(x)[1],
        "macd_hist": compute_macd(x)[2],
    }, index=x.index))
)

macd_results = macd_results.reset_index(level=0, drop=True)

features_df[["macd", "macd_signal", "macd_hist"]] = macd_results[
    ["macd", "macd_signal", "macd_hist"]
]

display(
    features_df[
        [
            "date",
            "instrument",
            "close",
            "macd",
            "macd_signal",
            "macd_hist"
        ]
    ].head(70)
)

## 2.5 Time-Series Dependence Features

Financial time series are not independent through time.

In particular:
- returns may exhibit short-term momentum or mean reversion,
- volatility tends to cluster,
- and trends may persist for several consecutive periods.

We therefore construct features designed to capture temporal dependence structures.

### Rolling Return Autocorrelation

We compute rolling autocorrelation of returns to measure:
- short-term trend persistence,
- or mean-reversion effects.

### Absolute Return Autocorrelation

We also compute autocorrelation of absolute returns:

$$
Corr(|r_t|, |r_{t-1}|)
$$

which is a classical proxy for volatility clustering.

### Trend Persistence

Finally, we compute rolling proportions of positive-return days in order to measure the persistence of directional trends.

In [ ]:
# ============================================================
# Rolling Return Autocorrelation
# ============================================================

AUTOCORR_WINDOWS = [20, 60]

for window in AUTOCORR_WINDOWS:

    features_df[f"autocorr_return_{window}d"] = (
        features_df
        .groupby("instrument")["log_return"]
        .transform(
            lambda x: x.rolling(window).corr(x.shift(1))
        )
    )

display(
    features_df[
        [
            "date",
            "instrument",
            "log_return",
            "autocorr_return_20d",
            "autocorr_return_60d"
        ]
    ].head(80)
)

In [ ]:
# ============================================================
# Absolute Return Autocorrelation
# ============================================================

features_df["abs_log_return"] = (
    features_df["log_return"].abs()
)

for window in AUTOCORR_WINDOWS:

    features_df[f"autocorr_abs_return_{window}d"] = (
        features_df
        .groupby("instrument")["abs_log_return"]
        .transform(
            lambda x: x.rolling(window).corr(x.shift(1))
        )
    )

display(
    features_df[
        [
            "date",
            "instrument",
            "abs_log_return",
            "autocorr_abs_return_20d",
            "autocorr_abs_return_60d"
        ]
    ].head(80)
)

In [ ]:
# ============================================================
# Trend Persistence Features
# ============================================================

# Positive-return indicator
features_df["positive_return"] = (
    (features_df["log_return"] > 0).astype(int)
)

PERSISTENCE_WINDOWS = [10, 20]

for window in PERSISTENCE_WINDOWS:

    features_df[f"positive_return_ratio_{window}d"] = (
        features_df
        .groupby("instrument")["positive_return"]
        .transform(
            lambda x: x.rolling(window).mean()
        )
    )

display(
    features_df[
        [
            "date",
            "instrument",
            "positive_return",
            "positive_return_ratio_10d",
            "positive_return_ratio_20d"
        ]
    ].head(70)
)

## 2.6 Cross-Sectional Features

So far, all features were computed independently for each instrument through time.

We now introduce **cross-sectional features**, which compare instruments within the Energy asset class at a given date.

These features attempt to capture:
- relative strength,
- relative volatility,
- sector dispersion,
- and intra-sector positioning.

Cross-sectional information is particularly important in multi-asset systematic trading because the attractiveness of a signal may depend not only on the state of one asset, but also on its position relative to the rest of the sector.

All cross-sectional features are computed date-by-date across the Energy universe.

In [ ]:
# ============================================================
# Cross-Sectional Momentum Rank
# ============================================================

features_df["momentum_rank_20d"] = (
    features_df
    .groupby("date")["momentum_20d"]
    .rank(pct=True)
)

features_df["momentum_rank_60d"] = (
    features_df
    .groupby("date")["momentum_60d"]
    .rank(pct=True)
)

display(
    features_df[
        [
            "date",
            "instrument",
            "momentum_20d",
            "momentum_rank_20d",
            "momentum_60d",
            "momentum_rank_60d"
        ]
    ]
    .sort_values(["date", "momentum_rank_20d"])
    .head(70)
)

In [ ]:
# ============================================================
# Relative Volatility Features
# ============================================================

# Sector average volatility by date
sector_avg_vol_20d = (
    features_df
    .groupby("date")["realized_vol_20d"]
    .transform("mean")
)

sector_avg_vol_60d = (
    features_df
    .groupby("date")["realized_vol_60d"]
    .transform("mean")
)

# Relative volatility
features_df["relative_vol_20d"] = (
    features_df["realized_vol_20d"]
    / sector_avg_vol_20d
)

features_df["relative_vol_60d"] = (
    features_df["realized_vol_60d"]
    / sector_avg_vol_60d
)

display(
    features_df[
        [
            "date",
            "instrument",
            "realized_vol_20d",
            "relative_vol_20d",
            "realized_vol_60d",
            "relative_vol_60d"
        ]
    ].head(70)
)

In [ ]:
# ============================================================
# Sector Dispersion Features
# ============================================================

# Cross-sectional std of momentum across assets
features_df["sector_momentum_dispersion_20d"] = (
    features_df
    .groupby("date")["momentum_20d"]
    .transform("std")
)

features_df["sector_momentum_dispersion_60d"] = (
    features_df
    .groupby("date")["momentum_60d"]
    .transform("std")
)

# Cross-sectional std of volatility across assets
features_df["sector_vol_dispersion_20d"] = (
    features_df
    .groupby("date")["realized_vol_20d"]
    .transform("std")
)

display(
    features_df[
        [
            "date",
            "instrument",
            "sector_momentum_dispersion_20d",
            "sector_momentum_dispersion_60d",
            "sector_vol_dispersion_20d"
        ]
    ].head(70)
)

In [ ]:
# ============================================================
# Relative Momentum vs Sector Mean
# ============================================================

sector_mean_momentum_20d = (
    features_df
    .groupby("date")["momentum_20d"]
    .transform("mean")
)

features_df["relative_momentum_20d"] = (
    features_df["momentum_20d"]
    - sector_mean_momentum_20d
)

display(
    features_df[
        [
            "date",
            "instrument",
            "momentum_20d",
            "relative_momentum_20d"
        ]
    ]
    .sort_values(["date", "relative_momentum_20d"])
    .head(70)
)

## 2.7 Feature Set Summary

Before moving to latent regime features, we summarize the feature set created so far and inspect the amount of missing data generated by rolling-window calculations.

In [ ]:
# ============================================================
# Feature Set Summary
# ============================================================

feature_columns = [
    col for col in features_df.columns
    if col not in [
        "date", "instrument", "open", "high", "low", "close",
        "volume", "open_interest"
    ]
]

print("Number of engineered features:", len(feature_columns))
print("\nFeature columns:")
for col in feature_columns:
    print("-", col)

missing_summary = (
    features_df[feature_columns]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .to_frame("missing_ratio")
)

display(missing_summary.head(30))

## 2.8 HMM Regime Features

We now add latent regime features using a Hidden Markov Model (HMM).

Unlike technical indicators, HMM features are not directly computed from prices. Instead, the HMM learns hidden market states from observable variables such as returns, volatility, and momentum.

To avoid look-ahead bias, the HMM will be fitted only on the training period for each instrument. The fitted model will then be used to infer regime probabilities for both the training and test periods.

In [ ]:
# ============================================================
# HMM Setup
# ============================================================

try:
    from hmmlearn.hmm import GaussianHMM
    print("hmmlearn is already installed.")
except ImportError:
    print("hmmlearn is not installed. Run the next cell:")
    print("!pip install hmmlearn")

### 2.8.1 Define HMM Inputs

We use a small set of interpretable variables to train the HMM:

- daily log return,
- 20-day realized volatility,
- 20-day momentum.

These variables allow the HMM to distinguish between calm, volatile, trending, and stressed market regimes.

In [ ]:
# ============================================================
# Define HMM input features
# ============================================================

HMM_FEATURES = [
    "log_return",
    "realized_vol_20d",
    "momentum_20d"
]

N_HMM_STATES = 3

print("HMM input features:", HMM_FEATURES)
print("Number of HMM regimes:", N_HMM_STATES)

### 2.8.2 Define Train/Test Split for HMM

Before fitting the HMM, we define a chronological train/test split using the primary signal dates.

For each instrument, the first 80% of signal dates are assigned to the training period, while the last 20% are reserved as the final out-of-sample test period.

The HMM is then fitted only using market data up to the last training signal date for each instrument.

In [ ]:
# ============================================================
# Define chronological 80/20 split from primary signals
# ============================================================

TEST_SIZE = 0.20

signal_split_info = {}

for instrument in ENERGY_INSTRUMENTS:
    
    signal_dates = (
        signals_long
        .loc[signals_long["instrument"] == instrument, "date"]
        .sort_values()
        .drop_duplicates()
        .reset_index(drop=True)
    )
    
    split_idx = int(len(signal_dates) * (1 - TEST_SIZE))
    
    train_dates = signal_dates.iloc[:split_idx]
    test_dates = signal_dates.iloc[split_idx:]
    
    signal_split_info[instrument] = {
        "n_dates": len(signal_dates),
        "n_train_dates": len(train_dates),
        "n_test_dates": len(test_dates),
        "train_start": train_dates.min(),
        "train_end": train_dates.max(),
        "test_start": test_dates.min(),
        "test_end": test_dates.max()
    }

split_summary = pd.DataFrame(signal_split_info).T
display(split_summary)

### 2.8.3 Train One HMM per Instrument

We fit one Gaussian HMM per energy instrument.

Each HMM is trained only on the training period and then used to infer latent regimes over the full sample.

The inferred regimes will later be used as additional features for the metamodel.

In [ ]:
# ============================================================
# Initialize HMM outputs
# ============================================================

from sklearn.preprocessing import StandardScaler

features_df["hmm_regime"] = np.nan

for state in range(N_HMM_STATES):
    features_df[f"hmm_prob_state_{state}"] = np.nan

In [ ]:
# ============================================================
# Train HMMs and infer regimes
# ============================================================

hmm_models = {}
hmm_scalers = {}

for instrument in ENERGY_INSTRUMENTS:
    
    print(f"\nTraining HMM for {instrument}...")
    
    # --------------------------------------------------------
    # Instrument subset
    # --------------------------------------------------------
    
    instrument_df = (
        features_df
        .loc[features_df["instrument"] == instrument]
        .sort_values("date")
        .copy()
    )
    
    # --------------------------------------------------------
    # Keep only rows with valid HMM features
    # --------------------------------------------------------
    
    instrument_df = instrument_df.dropna(subset=HMM_FEATURES)
    
    # --------------------------------------------------------
    # Train/test cutoff
    # --------------------------------------------------------
    
    train_end_date = signal_split_info[instrument]["train_end"]
    
    train_mask = instrument_df["date"] <= train_end_date
    
    train_df = instrument_df.loc[train_mask]
    
    # --------------------------------------------------------
    # Extract matrices
    # --------------------------------------------------------
    
    X_train = train_df[HMM_FEATURES].values
    X_full = instrument_df[HMM_FEATURES].values
    
    # --------------------------------------------------------
    # Standardize using TRAIN ONLY
    # --------------------------------------------------------
    
    scaler = StandardScaler()
    
    scaler.fit(X_train)
    
    X_train_scaled = scaler.transform(X_train)
    X_full_scaled = scaler.transform(X_full)
    
    hmm_scalers[instrument] = scaler
    
    # --------------------------------------------------------
    # Fit Gaussian HMM
    # --------------------------------------------------------
    
    hmm = GaussianHMM(
        n_components=N_HMM_STATES,
        covariance_type="full",
        n_iter=300,
        random_state=42
    )
    
    hmm.fit(X_train_scaled)
    
    hmm_models[instrument] = hmm
    
    # --------------------------------------------------------
    # Infer regimes on full sample
    # --------------------------------------------------------
    
    hidden_states = hmm.predict(X_full_scaled)
    
    state_probs = hmm.predict_proba(X_full_scaled)
    
    # --------------------------------------------------------
    # Store results
    # --------------------------------------------------------
    
    features_df.loc[instrument_df.index, "hmm_regime"] = hidden_states
    
    for state in range(N_HMM_STATES):
        
        features_df.loc[
            instrument_df.index,
            f"hmm_prob_state_{state}"
        ] = state_probs[:, state]
    
    print("Done.")

### 2.8.4 Inspect HMM Regime Features

We now inspect the HMM regime features to verify that:

- each observation is assigned a regime,
- regime probabilities sum to one,
- regimes are used across instruments,
- and the inferred states can later be interpreted economically.

In [ ]:
# ============================================================
# Inspect HMM feature columns
# ============================================================

hmm_cols = [
    "hmm_regime",
    "hmm_prob_state_0",
    "hmm_prob_state_1",
    "hmm_prob_state_2"
]

display(
    features_df[
        ["date", "instrument"] + HMM_FEATURES + hmm_cols
    ]
    .dropna(subset=hmm_cols)
    .head(20)
)

In [ ]:
# ============================================================
# Check HMM probability sums
# ============================================================

features_df["hmm_prob_sum"] = (
    features_df["hmm_prob_state_0"]
    + features_df["hmm_prob_state_1"]
    + features_df["hmm_prob_state_2"]
)

display(
    features_df[
        ["date", "instrument", "hmm_prob_sum"]
    ]
    .dropna()
    .head(20)
)

print(
    "Min probability sum:",
    features_df["hmm_prob_sum"].min()
)

print(
    "Max probability sum:",
    features_df["hmm_prob_sum"].max()
)

## 2.9 Merge Features with Primary Signals

We now merge the engineered feature set with the primary trading signals.

The resulting dataframe contains one row per `(date, instrument)` where the primary model provides a signal. This will become the base dataset used for triple-barrier labeling and later metamodel training.

At this stage, we keep both active signals (`+1`, `-1`) and inactive signals (`0`). The active signals will be used for labeling trades, while inactive signals ????.

In [ ]:
# ============================================================
# Merge engineered features with primary signals
# ============================================================

model_base_df = (
    signals_long
    .merge(
        features_df,
        on=["date", "instrument"],
        how="left"
    )
    .sort_values(["instrument", "date"])
    .reset_index(drop=True)
)

print("Model base dataframe shape:", model_base_df.shape)

display(
    model_base_df[
        [
            "date",
            "instrument",
            "primary_signal",
            "close",
            "log_return",
            "momentum_20d",
            "realized_vol_20d",
            "hmm_regime",
            "hmm_prob_state_0",
            "hmm_prob_state_1",
            "hmm_prob_state_2"
        ]
    ].head(20)
)

In [ ]:
# ============================================================
# Check merge quality
# ============================================================

print("Missing close after merge:", model_base_df["close"].isna().sum())
print("Missing log_return after merge:", model_base_df["log_return"].isna().sum())
print("Missing HMM regime after merge:", model_base_df["hmm_regime"].isna().sum())

print("\nPrimary signal distribution:")
display(
    model_base_df["primary_signal"]
    .value_counts()
    .sort_index()
)

print("\nPrimary signal distribution by instrument:")
display(
    pd.crosstab(
        model_base_df["instrument"],
        model_base_df["primary_signal"]
    )
)

In [ ]:
# Rows where signals exist but no matching OHLCV/features were found
missing_feature_rows = model_base_df[model_base_df["close"].isna()].copy()

print("Number of rows with missing features:", len(missing_feature_rows))

display(
    missing_feature_rows[
        ["date", "instrument", "primary_signal"]
    ]
    .sort_values(["instrument", "date"])
    .head(50)
)

In [ ]:
# Count missing feature rows by instrument
display(
    missing_feature_rows["instrument"]
    .value_counts()
)

In [ ]:
# Compare available dates between signals and features for each instrument
for instrument in ENERGY_INSTRUMENTS:
    
    signal_dates = set(
        signals_long.loc[
            signals_long["instrument"] == instrument,
            "date"
        ]
    )
    
    feature_dates = set(
        features_df.loc[
            features_df["instrument"] == instrument,
            "date"
        ]
    )
    
    missing_dates = sorted(signal_dates - feature_dates)
    
    print(f"\n{instrument}")
    print("Missing dates:", len(missing_dates))
    print(missing_dates[:10])

In [ ]:
# ============================================================
# Drop signal rows without matching OHLCV/features
# ============================================================

model_base_df = (
    model_base_df
    .dropna(subset=["close"])
    .copy()
    .reset_index(drop=True)
)

print("Clean model base dataframe shape:", model_base_df.shape)

print("Remaining missing close:", model_base_df["close"].isna().sum())
print("Remaining missing HMM regime:", model_base_df["hmm_regime"].isna().sum())

## Summary Before Phase 3: Dataset Construction and Feature Engineering

At this stage, we have completed the full feature engineering pipeline and built the base dataset for the metamodel.

The workflow implemented so far is summarized below:

```text
Raw OHLCV data
        ↓
Feature Engineering
    - Return and momentum features
    - Volatility features
    - Technical indicators
    - Time-series features
    - Cross-sectional features
        ↓
HMM regime features
        ↓
Merge with primary trading signals
        ↓
Base metamodel dataset
```

### Features Created

The engineered features currently include several categories:

**Technical features**
- Momentum (multiple horizons)
- Realized volatility
- Rolling means
- RSI
- MACD
- Price z-score
- Volume z-score
- Open interest momentum

**Time-series features**
- Return autocorrelation
- Absolute return autocorrelation
- Trend persistence
- Volatility persistence

**Cross-sectional features**
- Relative momentum
- Momentum rank
- Relative volatility
- Sector dispersion measures

**Latent regime features (HMM)**
- `hmm_regime`
- `hmm_prob_state_0`
- `hmm_prob_state_1`
- `hmm_prob_state_2`

The HMM was trained independently for each energy instrument using:

- log returns
- realized volatility
- momentum

To avoid look-ahead bias:

- the HMM was fitted only on the training period,
- standardization was fitted only on training data,
- the trained HMM was then used to infer regimes over the full sample.

---

### Base Metamodel Dataset

The engineered features were merged with the primary trading signals to create:

```python
model_base_df
```

This dataset now contains:

- date
- instrument
- primary signal
- market variables
- engineered features
- HMM regime features

Each row corresponds to one `(date, instrument)` pair where a primary signal exists.

---

### Data Cleaning

During the merge process, some rows contained missing values because certain signal dates corresponded to market holidays or non-trading days.

Examples include:

- Martin Luther King Day
- Presidents’ Day
- Thanksgiving
- Good Friday

These observations were removed because no OHLCV data existed for those dates.

---

### Train/Test Protocol

A chronological train/test split has already been defined and is now frozen.

The split was created from the primary signal dataset:

- first 80% → training period
- last 20% → final out-of-sample period

This split will remain unchanged for:

- HMM training
- Triple-barrier labeling
- Hyperparameter tuning
- Model training
- Final evaluation

No future information has been used during feature construction or HMM fitting.

The next step is to construct the target variable using the Triple-Barrier labeling framework.

# Phase 3 — Triple Barrier Labeling: Methodological Decisions and Design Notes

## Objective

The purpose of Phase 3 is to transform the active primary trading signals into supervised learning labels for the meta-model.

The primary model already predicts the trade direction:

- `+1` → long trade
- `-1` → short trade
- `0` → no trade

The meta-model does **not** try to predict market direction.

Instead, it learns:

$$
P(\text{Trade is profitable} \mid \text{Market Context})
$$

Its role is therefore:

- filter bad trades
- keep good trades
- potentially adjust confidence or position sizing later

---

# 1. Why inactive signals (`0`) create a problem

Our merged dataset currently contains:

```text
(date, instrument)
```

rows for all observations:

```text
date      instrument      primary_signal
-----------------------------------------
t1         CL1S                 +1
t2         CL1S                  0
t3         CL1S                 -1
t4         NG1S                  0
...
```

A `primary_signal = 0` means:

```text
No trade proposed by the primary model
```

Therefore there is:

- no entry price
- no trade
- no trade outcome
- no profit-taking event
- no stop-loss event

Thus we cannot directly assign:

```python
label = 0 or 1
```

because there is no trade to evaluate.

---

## Options considered for handling `0`

### Option 1 — Convert `0` into previous signal

Example:

```text
+1 → 0 → 0 → +1
```

becomes:

```text
+1 → +1 → +1 → +1
```

Problems:

- creates artificial trades
- assumes the primary model wanted to keep trading
- modifies the original signal process

Decision:

❌ Rejected

---

### Option 2 — Infer direction from local trend

Example:

Use recent momentum:

```python
if momentum > 0:
    signal=+1
else:
    signal=-1
```

Problems:

- effectively creates a new primary model
- no longer evaluates the provided signals
- breaks the meta-labeling philosophy

Decision:

❌ Rejected

---

### Option 3 — Exclude inactive signals from meta-labeling

Create:

```python
meta_df = df[df.primary_signal !=0]
```

while keeping the original dataframe intact.

Decision:

✅ Selected

---

Important:

We are NOT deleting information.

We keep:

```python
df_full
```

containing:

```text
+1
-1
0
```

The inactive observations can still contribute to:

- volatility estimates
- rolling statistics
- HMM/GMM regime detection
- feature construction
- descriptive analysis

Only:

```python
meta_df
```

will be used for supervised labels.

---

# 2. Sample size issue

Removing inactive signals dramatically reduces observations.

Example:

```text
HO1S:

-1 : 10
0 : 582
+1 : 53
```

Active trades:

```text
63 only
```

Training one independent model per instrument would therefore become unstable.

---

# 3. Global (pooled) meta-model approach

Instead of:

```text
model_CL1S
model_HO1S
model_NG1S
...
```

we train:

```text
ONE global meta-model
```

across all instruments.

---

## Why this helps

Instead of learning only from:

```text
HO1S → 63 observations
```

the model learns from:

```text
HO1S
CL1S
NG1S
RB1S
ES1S
...
```

giving potentially thousands of trade observations.

This allows:

- more stable estimation
- reduced overfitting
- better generalization

---

# 4. How the model distinguishes instruments

Each observation remains:

```text
(date, instrument)
```

Example:

```text
date      instrument   signal   vol20   mom20   label
-------------------------------------------------------
2020      CL1S          +1      0.15     0.2      1
2020      NG1S          -1      0.60    -0.1      0
2020      HO1S          +1      0.35     0.1      1
```

We include:

```text
instrument
asset_class
primary_signal
```

as features.

The model can therefore learn:

```text
"NG1S short trades perform well under high volatility"

or

"HO1S behaves differently"
```

Instrument information will likely be encoded via:

- one-hot encoding
- categorical encoding

---

# 5. Time ordering and dataframe structure

Current dataframe:

```text
CL1S 2020→2022
HO1S 2020→2022
RB1S 2020→2022
...
```

Dates restart for each instrument.

This is NOT problematic for ML.

However:

Triple-barrier calculations MUST be performed separately for each instrument.

Therefore:

```python
df = df.sort_values(
    ["instrument","date"]
)

df.groupby(
    "instrument"
)
```

will be used.

Otherwise:

future windows could accidentally jump between instruments.

---

Rule:

```text
Labeling:
groupby(instrument)

Model training:
pooled across instruments
```

---

# 6. Train/Validation/Test split

We already fixed the original temporal boundaries earlier.

Important:

These boundaries MUST remain unchanged.

Even after removing rows or constructing labels we do NOT recompute:

```text
20%
60%
20%
```

splits.

Instead we keep:

```text
Original train end date
Original validation dates
Original test dates
```

and apply:

```python
train = data[
date < train_end
]

validation = ...

test = ...
```

This avoids look-ahead bias.

---

# 7. Triple Barrier parameters

Three parameters must be selected:

---

## Horizontal barriers

Profit-taking:

\[
PT
\]

Stop-loss:

\[
SL
\]

---

## Vertical barrier

Maximum holding period:

\[
h
\]

---

# 8. Possible choices for PT and SL

---

### Option A

Fixed returns

Example:

```python
PT=3%
SL=2%
```

Advantages:

- simple

Problems:

- ignores volatility differences

Decision:

❌ Probably not ideal

---

### Option B

Volatility-scaled barriers

Example:

$$
Upper
=
P_t(1+k_{PT}\sigma_t)
$$

$$
Lower
=
P_t(1-k_{SL}\sigma_t)
$$

Advantages:

- adapts to each instrument
- comparable across markets

Decision:

✅ Preferred

---

Potential parameter grid:

```python
pt_grid=[1,1.5,2]

sl_grid=[1,1.5,2]
```

---

# 9. Horizon selection

Possible horizons:

```python
h=[5,10,20]
```

---

Option A:

Different horizon per instrument

Advantages:

- potentially more realistic

Problems:

- more complexity
- labels less comparable

Decision:

❌ Not initially

---

Option B:

Common horizon

Advantages:

- simpler
- easier interpretation

Decision:

✅ Preferred initially

---

# 10. Handling the vertical barrier outcome

Case:

No PT and no SL reached before:

\[
h
\]

---

Option A

Always:

```python
label=0
```

Advantages:

- strict

Problems:

- can create too many negatives

Decision:

❌ Possibly too harsh

---

Option B

Use return at horizon

Compute:

$$
r_h
=
\frac{P_{t+h}}{P_t}-1
$$

Adjusted for trade direction:

$$
TradeReturn
=
Signal\times r_h
$$

Then:

```python
if TradeReturn>0:
    label=1
else:
    label=0
```

Advantages:

- keeps information
- avoids excessive data loss
- economically meaningful

Decision:

✅ Preferred

---

Option C

Drop observations

```python
label=NaN
```

Advantages:

- avoids ambiguous cases

Problems:

- further reduces sample size

Decision:

❌ Not preferred

---

# Preliminary selected approach

Dataset:

```python
meta_df=df[
df.primary_signal!=0
]
```

Model:

```text
One pooled global model
```

Distinguish instruments using:

```text
instrument
asset_class
primary_signal
```

Labeling:

```text
groupby(instrument)
```

Barriers:

```text
Volatility scaled PT/SL
```

Horizon:

```text
Common h
```

Vertical barrier:

```text
Use sign-adjusted return at h
```

Temporal split:

```text
Keep original train/validation/test dates unchanged
```

These choices remain preliminary and can later be compared empirically during model development and validation.

# Phase 3 — Triple-Barrier Meta-Labeling

We now construct the supervised target used by the meta-model.

The primary model already provides a trade direction:

- `+1`: long trade
- `-1`: short trade
- `0`: no trade

The meta-model should not create new trades. Its purpose is to decide whether an existing active signal should be trusted or filtered out.

Therefore, triple-barrier labels are only computed for active primary signals (`+1` or `-1`). Inactive signals are kept in the full dataset for feature construction and regime analysis, but they are excluded from the meta-labeling target because no trade is proposed by the primary model.

We keep two datasets:

- `model_base_df`: full dataset after feature merge
- `meta_df`: active-signal dataset used for triple-barrier labeling

In [ ]:
# ============================================================
# Phase 3.1 — Construct Active-Signal Meta-Labeling Dataset
# ============================================================

# Keep a full copy for reference, feature analysis, and possible regime work
full_model_df = model_base_df.copy()

# Meta-labeling is only meaningful when the primary model proposes a trade
meta_df = (
    model_base_df
    .loc[model_base_df["primary_signal"] != 0]
    .copy()
    .sort_values(["instrument", "date"])
    .reset_index(drop=True)
)

print("Full model dataframe shape:", full_model_df.shape)
print("Meta-labeling dataframe shape:", meta_df.shape)

print("\nPrimary signal distribution in full dataset:")
display(
    full_model_df["primary_signal"]
    .value_counts()
    .sort_index()
)

print("\nPrimary signal distribution in meta-labeling dataset:")
display(
    meta_df["primary_signal"]
    .value_counts()
    .sort_index()
)

print("\nActive signal distribution by instrument:")
display(
    pd.crosstab(
        meta_df["instrument"],
        meta_df["primary_signal"]
    )
)

## 3.2 Temporal Structure Check

Before computing triple-barrier labels, we verify that the active-signal dataset is ordered by instrument and date.

This matters because triple-barrier labeling uses future prices. The future path of a trade must always be taken from the same instrument, never from the next instrument in the stacked dataframe.

Therefore, all labeling functions will be applied using:

```python
groupby("instrument")

In [ ]:

# ============================================================
# Phase 3.2 — Temporal Structure Check
# ============================================================

for instrument, g in meta_df.groupby("instrument"):
    is_sorted = g["date"].is_monotonic_increasing
    print(f"{instrument}: sorted by date = {is_sorted}, n_active_signals = {len(g)}")

print("\nDate range by instrument:")
display(
    meta_df
    .groupby("instrument")["date"]
    .agg(["min", "max", "count"])
)

## 3.3 Choice of Volatility Estimator

The triple-barrier method requires a volatility estimate to define the width of the profit-taking and stop-loss barriers.

Several volatility estimators were considered.

### Option 1 — Rolling realized volatility

\[
\sigma_t
=
Std(r_{t-n+1},...,r_t)
\]

Advantages:

- simple
- widely used
- easy interpretation
- computationally inexpensive

Disadvantages:

- all observations receive equal weight
- does not explicitly capture volatility clustering

Decision for baseline:

✅ Selected initially

We use:

```python
realized_vol_20d
```

This choice provides a compromise between:

- responsiveness
- stability
- interpretability

and is consistent with an intermediate holding horizon.

---

### Option 2 — EWMA volatility

\[
\sigma_t^2
=
\lambda\sigma_{t-1}^2
+
(1-\lambda)r_t^2
\]

Advantages:

- larger weight on recent observations
- captures changing volatility environments

Potential future extension:

⏳ Consider later

---

### Option 3 — GARCH volatility

Advantages:

- captures volatility clustering

Disadvantages:

- parameter estimation required
- increased complexity

Potential future extension:

⏳ Consider later

---

### Option 4 — Range-based volatility (ATR / Parkinson / Garman-Klass)

Advantages:

- uses OHLC information
- potentially more informative for futures markets

Potential future extension:

⏳ Consider later

---

For the initial implementation we adopt:

```python
vol_col = "realized_vol_20d"
```

to establish a simple and robust baseline before exploring more sophisticated alternatives.

## 3.4 Initial Triple-Barrier Parameters

We define an initial baseline specification for the triple-barrier method.

The selected values are intentionally simple and symmetric.

Initial choices:

- Holding horizon: `h = 10`
- Profit-taking multiplier: `PT = 1.5`
- Stop-loss multiplier: `SL = 1.5`
- Volatility estimator: `realized_vol_20d`

These choices are not assumed to be optimal and may later be compared against alternative specifications using validation performance.

In [ ]:
# ============================================================
# Phase 3.4 — Initial Triple Barrier Parameters
# ============================================================

# Entry price used for trades
price_col = "close"

# Volatility estimate
vol_col = "realized_vol_20d"

# Vertical barrier
HORIZON = 10

# Horizontal barriers
PT_MULT = 1.5
SL_MULT = 1.5


print("Selected baseline parameters")
print("--------------------------------")
print("Price column:", price_col)
print("Volatility:", vol_col)
print("Holding horizon:", HORIZON)
print("PT multiplier:", PT_MULT)
print("SL multiplier:", SL_MULT)

## 3.5 Triple-Barrier Labeling Function

We now implement the triple-barrier labeling rule.

For each active trade:

- if the primary signal is `+1`, the trade is long;
- if the primary signal is `-1`, the trade is short.

The barriers are computed using the entry price and the volatility estimate at the trade date.

For a long trade:

- hitting the upper barrier first gives `meta_label = 1`;
- hitting the lower barrier first gives `meta_label = 0`.

For a short trade:

- hitting the lower barrier first gives `meta_label = 1`;
- hitting the upper barrier first gives `meta_label = 0`.

If neither horizontal barrier is reached before the vertical barrier, we use the sign-adjusted return at horizon \(h\):

$$
\text{trade return}
=
\text{primary signal}
\times
\left(
\frac{P_{t+h}}{P_t}-1
\right)
$$

and assign:

$$
\text{meta label}
=
1\{\text{trade return}>0\}
$$

In [ ]:
# ============================================================
# Phase 3.5 — Triple Barrier Labeling Function
# ============================================================

def apply_triple_barrier_to_instrument(
    instrument_df,
    price_col,
    vol_col,
    horizon=10,
    pt_mult=1.5,
    sl_mult=1.5
):
    """
    Apply triple-barrier labeling to one instrument at a time.

    Parameters
    ----------
    instrument_df : pd.DataFrame
        DataFrame containing only one instrument, sorted by date.
    price_col : str
        Column used as the trade entry price.
    vol_col : str
        Backward-looking volatility column used to scale barriers.
    horizon : int
        Maximum holding period in number of rows/trading days.
    pt_mult : float
        Profit-taking multiplier.
    sl_mult : float
        Stop-loss multiplier.

    Returns
    -------
    pd.DataFrame
        Same dataframe with triple-barrier outputs added.
    """

    df_inst = instrument_df.copy().sort_values("date").reset_index(drop=True)

    meta_labels = []
    event_types = []
    exit_dates = []
    exit_prices = []
    trade_returns = []

    prices = df_inst[price_col].values
    vols = df_inst[vol_col].values
    signals = df_inst["primary_signal"].values
    dates = df_inst["date"].values

    n = len(df_inst)

    for i in range(n):

        entry_price = prices[i]
        vol_t = vols[i]
        signal = signals[i]

        # If volatility or price is missing, we cannot define reliable barriers
        if pd.isna(entry_price) or pd.isna(vol_t) or vol_t <= 0:
            meta_labels.append(np.nan)
            event_types.append("missing_input")
            exit_dates.append(pd.NaT)
            exit_prices.append(np.nan)
            trade_returns.append(np.nan)
            continue

        # If there is not enough future data to reach the vertical barrier,
        # we cannot label the trade cleanly
        end_i = min(i + horizon, n - 1)

        if end_i == i:
            meta_labels.append(np.nan)
            event_types.append("no_future_data")
            exit_dates.append(pd.NaT)
            exit_prices.append(np.nan)
            trade_returns.append(np.nan)
            continue

        upper_barrier = entry_price * (1 + pt_mult * vol_t)
        lower_barrier = entry_price * (1 - sl_mult * vol_t)

        label = None
        event_type = None
        exit_i = None

        # Scan the future path until the vertical barrier
        for j in range(i + 1, end_i + 1):

            price_j = prices[j]

            if signal == 1:
                # Long trade
                if price_j >= upper_barrier:
                    label = 1
                    event_type = "profit_taking"
                    exit_i = j
                    break

                elif price_j <= lower_barrier:
                    label = 0
                    event_type = "stop_loss"
                    exit_i = j
                    break

            elif signal == -1:
                # Short trade
                if price_j <= lower_barrier:
                    label = 1
                    event_type = "profit_taking"
                    exit_i = j
                    break

                elif price_j >= upper_barrier:
                    label = 0
                    event_type = "stop_loss"
                    exit_i = j
                    break

        # If no horizontal barrier is hit, use vertical barrier outcome
        if label is None:
            exit_i = end_i
            exit_price = prices[exit_i]

            realized_trade_return = signal * (exit_price / entry_price - 1)

            label = int(realized_trade_return > 0)
            event_type = "vertical_barrier"

        else:
            exit_price = prices[exit_i]
            realized_trade_return = signal * (exit_price / entry_price - 1)

        meta_labels.append(label)
        event_types.append(event_type)
        exit_dates.append(dates[exit_i])
        exit_prices.append(exit_price)
        trade_returns.append(realized_trade_return)

    df_inst["meta_label"] = meta_labels
    df_inst["tb_event_type"] = event_types
    df_inst["tb_exit_date"] = exit_dates
    df_inst["tb_exit_price"] = exit_prices
    df_inst["tb_trade_return"] = trade_returns

    return df_inst

## 3.6 Apply Triple-Barrier Labeling

The labeling function is now applied separately to each instrument.

This ensures that:

- future paths are computed only within the same instrument;
- dates remain temporally consistent;
- barrier calculations use instrument-specific volatility dynamics.

The resulting dataset becomes the final supervised dataset used later for meta-model training.

In [ ]:
# ============================================================
# Phase 3.6 — Apply Triple Barrier Labeling
# ============================================================

labeled_groups = []

for instrument, g in meta_df.groupby("instrument"):
    g = g.copy()
    g["instrument"] = instrument  # keep instrument explicitly as a column
    
    labeled_g = apply_triple_barrier_to_instrument(
        g,
        price_col=price_col,
        vol_col=vol_col,
        horizon=HORIZON,
        pt_mult=PT_MULT,
        sl_mult=SL_MULT
    )
    
    labeled_g["instrument"] = instrument
    labeled_groups.append(labeled_g)

meta_labeled_df = (
    pd.concat(labeled_groups, axis=0)
    .sort_values(["instrument", "date"])
    .reset_index(drop=True)
)

print("Final labeled dataset shape:")
print(meta_labeled_df.shape)

print("\nColumns check:")
print("instrument" in meta_labeled_df.columns)

print("\nMeta-label distribution:")
display(
    meta_labeled_df["meta_label"]
    .value_counts(dropna=False)
)

print("\nBarrier event distribution:")
display(
    meta_labeled_df["tb_event_type"]
    .value_counts(dropna=False)
)

In [ ]:
display(
    meta_df[
        meta_labeled_df["tb_event_type"]=="no_future_data"
    ][
        [
            "instrument",
            "date",
            "primary_signal"
        ]
    ]
)

## 3.7 Remove Unlabelable Observations

A very small number of observations occur at the end of the available sample and do not contain enough future information to reach the full vertical barrier horizon.

These observations cannot be labeled consistently and are therefore removed.

This does not introduce information leakage because the future path required for label construction simply does not exist.

In [ ]:
# ============================================================
# Phase 3.7 — Remove Unlabelable Observations
# ============================================================

n_before = len(meta_labeled_df)

meta_labeled_df = (
    meta_labeled_df
    .dropna(subset=["meta_label"])
    .reset_index(drop=True)
)

n_after = len(meta_labeled_df)

print("Rows before cleaning:", n_before)
print("Rows after cleaning:", n_after)

print(
    f"\nRemoved observations: {n_before-n_after}"
)

print(
    f"Percentage removed: "
    f"{100*(n_before-n_after)/n_before:.3f}%"
)

In [ ]:
# ============================================================
# Phase 3.8 — Label Distribution by Instrument
# ============================================================

print("Meta-label distribution by instrument:")

display(
    pd.crosstab(
        meta_labeled_df["instrument"],
        meta_labeled_df["meta_label"],
        margins=True
    )
)

print("\nPercentages by instrument:")

display(
    pd.crosstab(
        meta_labeled_df["instrument"],
        meta_labeled_df["meta_label"],
        normalize="index"
    ).round(3)
)

In [ ]:
# ============================================================
# Phase 3.9 — Barrier Event Distribution by Instrument
# ============================================================

print("Barrier event distribution by instrument:")

display(
    pd.crosstab(
        meta_labeled_df["instrument"],
        meta_labeled_df["tb_event_type"],
        margins=True
    )
)

print("\nPercentages by instrument:")

display(
    pd.crosstab(
        meta_labeled_df["instrument"],
        meta_labeled_df["tb_event_type"],
        normalize="index"
    ).round(3)
)